In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pathlib import Path
import ot
import scipy as sp
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from utils_mocap import LoadCloudPoint, DistanceProfile
from utils_mocap import compute_W_matrix_distance_matrix_input
from utils_mocap import plot_3d_points_and_connections

ImportError: cannot import name 'LoadCloudPoint' from 'utils' (c:\Users\Dzu\Desktop\motion_capture_ot\utils.py)

In [2]:
import random

random.seed(10)

lcp = LoadCloudPoint(filepath="datasets/csv_files/0005_Jogging001.csv")
source_pc, target_pc = lcp.get_two_random_point_cloud()

dp = DistanceProfile(source_pc, target_pc)
distance_matrix = dp.compute_L2_matrix()

Loaded point cloud data from datasets\csv_files\0005_Jogging001.csv, number of frames: 1377


In [3]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go

# Example mocap data (replace with your own)
np.random.seed(1)
mocap_data = source_pc

# Compute kNN
k = 2
nbrs = NearestNeighbors(n_neighbors=k+1).fit(mocap_data)
distances, indices = nbrs.kneighbors(mocap_data)
indices = indices[:,1:]

# Build edge lines
edge_x, edge_y, edge_z = [], [], []
for i in range(mocap_data.shape[0]):
    for j in indices[i]:
        p1 = mocap_data[i]
        p2 = mocap_data[j]
        edge_x += [p1[0], p2[0], None]
        edge_y += [p1[1], p2[1], None]
        edge_z += [p1[2], p2[2], None]

# Build figure
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode="lines",
    line=dict(width=2),
    hoverinfo="none"
))

fig.add_trace(go.Scatter3d(
    x=mocap_data[:,0],
    y=mocap_data[:,1],
    z=mocap_data[:,2],
    mode="markers",
    marker=dict(size=4),
    hoverinfo="none"
))

fig.update_layout(
    title="Interactive 3D kNN Mocap Graph",
    scene=dict(aspectmode="data"),
    width=900,
    height=700
)

fig.show()


# Find KNN matrix

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go

def plot_kth_neighbor_graph(points, k):
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(points)
    distances, indices = nbrs.kneighbors(points)
    indices = indices[:,1:]

    # Build edge lines
    edge_x, edge_y, edge_z = [], [], []
    for i in range(points.shape[0]):
        for j in indices[i]:
            p1 = points[i]
            p2 = points[j]
            edge_x += [p1[0], p2[0], None]
            edge_y += [p1[1], p2[1], None]
            edge_z += [p1[2], p2[2], None]

    # Build figure
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode="lines",
        line=dict(width=2),
        hoverinfo="none"
    ))

    fig.add_trace(go.Scatter3d(
        x=points[:,0],
        y=points[:,1],
        z=points[:,2],
        mode="markers",
        marker=dict(size=4),
        hoverinfo="none"
    ))

    fig.update_layout(
        title="Interactive 3D kNN Mocap Graph",
        scene=dict(aspectmode="data"),
        width=900,
        height=700
    )

    fig.show()

    # return knn adjency matrix
    knn_adj_matrix = np.zeros((points.shape[0], points.shape[0]))
    for i in range(points.shape[0]):
        for j in indices[i]:
            knn_adj_matrix[i, j] = 1

    # make an adjency matrix with each element being k-i where i is corresponding neighbor index
    knn_weighted_adj_matrix = np.zeros((points.shape[0], points.shape[0]))
    for i in range(points.shape[0]):
        for idx, j in enumerate(indices[i]):
            knn_weighted_adj_matrix[i, j] = k - idx
    return knn_adj_matrix, knn_weighted_adj_matrix


(adj_matrix, weighted_adj_matrix) = plot_kth_neighbor_graph(source_pc, k=6)



# weighted adj matrix

In [13]:
weighted_adj_matrix

array([[0., 2., 0., ..., 0., 0., 0.],
       [2., 0., 1., ..., 0., 0., 0.],
       [0., 2., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 2.],
       [0., 0., 0., ..., 1., 0., 2.],
       [0., 0., 0., ..., 1., 2., 0.]])

# Repeat for target

In [14]:
(adj_matrix, weighted_adj_matrix) = plot_kth_neighbor_graph(target_pc, k=6)

# Make interactions graph

In [21]:
# package into a data structure with {'src_index': ..., 'tar_index': ..., 'src_interactions': ..., 'tar_interactions': ...}

data_structure = {}

# list all indices of src points with {0.0: 0, 1.0: 1, 2.0: 2, ...}
data_structure['src_index'] = {float(i): i for i in range(source_pc.shape[0])}
# list all indices of tar points with {0.0: 0, 1.0: 1, 2.0: 2, ...}
data_structure['tar_index'] = {float(i): i for i in range(target_pc.shape[0])}

# list all interations for src points wit form [[0, np.int32(5)], [1, np.int32(3)], ...] with one listing for each weight i.e. a weight of 3 will have 3 listings
data_structure['src_interactions'] = []
for i in range(weighted_adj_matrix.shape[0]):
    for j in range(weighted_adj_matrix.shape[1]):
        weight = int(weighted_adj_matrix[i, j])
        for _ in range(weight):
            data_structure['src_interactions'].append([i, np.int32(j)])

# list all interations for tar points wit form [[0, np.int32(5)], [1, np.int32(3)], ...] with one listing for each weight i.e. a weight of 3 will have 3 listings
data_structure['tar_interactions'] = []
for i in range(weighted_adj_matrix.shape[0]):
    for j in range(weighted_adj_matrix.shape[1]):
        weight = int(weighted_adj_matrix[i, j])
        for _ in range(weight):
            data_structure['tar_interactions'].append([i, np.int32(j)])

data_structure

{'src_index': {0.0: 0,
  1.0: 1,
  2.0: 2,
  3.0: 3,
  4.0: 4,
  5.0: 5,
  6.0: 6,
  7.0: 7,
  8.0: 8,
  9.0: 9,
  10.0: 10,
  11.0: 11,
  12.0: 12,
  13.0: 13,
  14.0: 14,
  15.0: 15,
  16.0: 16,
  17.0: 17,
  18.0: 18,
  19.0: 19,
  20.0: 20,
  21.0: 21,
  22.0: 22,
  23.0: 23,
  24.0: 24,
  25.0: 25,
  26.0: 26,
  27.0: 27,
  28.0: 28,
  29.0: 29,
  30.0: 30,
  31.0: 31,
  32.0: 32,
  33.0: 33,
  34.0: 34,
  35.0: 35,
  36.0: 36,
  37.0: 37,
  38.0: 38,
  39.0: 39,
  40.0: 40,
  41.0: 41,
  42.0: 42,
  43.0: 43,
  44.0: 44,
  45.0: 45,
  46.0: 46,
  47.0: 47,
  48.0: 48,
  49.0: 49,
  50.0: 50,
  51.0: 51,
  52.0: 52},
 'tar_index': {0.0: 0,
  1.0: 1,
  2.0: 2,
  3.0: 3,
  4.0: 4,
  5.0: 5,
  6.0: 6,
  7.0: 7,
  8.0: 8,
  9.0: 9,
  10.0: 10,
  11.0: 11,
  12.0: 12,
  13.0: 13,
  14.0: 14,
  15.0: 15,
  16.0: 16,
  17.0: 17,
  18.0: 18,
  19.0: 19,
  20.0: 20,
  21.0: 21,
  22.0: 22,
  23.0: 23,
  24.0: 24,
  25.0: 25,
  26.0: 26,
  27.0: 27,
  28.0: 28,
  29.0: 29,
  30.0: 30,
  31.

In [22]:
data_structure.keys()

dict_keys(['src_index', 'tar_index', 'src_interactions', 'tar_interactions'])

# Shove in model

In [23]:
import dev.util as util
from dev.util import logger
import matplotlib.pyplot as plt
from model.GromovWassersteinLearning import GromovWassersteinLearning
from model.BAPG import process_interaction_data
import numpy as np
import pickle
import torch.optim as optim
from torch.optim import lr_scheduler
import time

In [26]:
time_GWEMBED = {}
time_BAPG = {}

node_accuracy_GWEMBED = {}
node_accuracy_BAPG = {}

nn = 'mc3'
n = 'test'
i = 0

n_nodes = ['test']
n_noises = 1

for n in n_nodes:
    for i in range(n_noises):
        time_GWEMBED[(n, i)] = []
        time_BAPG[(n, i)] = []
        node_accuracy_BAPG[(n, i)] = []
        node_accuracy_GWEMBED[(n, i)] = []

data_name = 'syn_{}_{}_{}'.format(nn, n, i)
result_folder = 'match_syn'
cost_type = ['cosine']
method = ['proximal']

util.makedirs(result_folder)

data_mc3 = data_structure


print(len(data_mc3['src_index']))
print(len(data_mc3['tar_index']))
print(len(data_mc3['src_interactions']))
print(len(data_mc3['tar_interactions']))

connects = np.zeros((len(data_mc3['src_index']), len(data_mc3['src_index'])))
for item in data_mc3['src_interactions']:
    connects[item[0], item[1]] += 1
plt.imshow(connects)
plt.savefig('{}/{}_src.png'.format(result_folder, data_name))
plt.close('all')

connects = np.zeros((len(data_mc3['tar_index']), len(data_mc3['tar_index'])))
for item in data_mc3['tar_interactions']:
    connects[item[0], item[1]] += 1
plt.imshow(connects)
plt.savefig('{}/{}_tar.png'.format(result_folder, data_name))
plt.close('all')

opt_dict = {'epochs': 5,
            'batch_size': 10000,
            'use_cuda': False,
            'strategy': 'soft',
            'beta': 1e-1,
            'outer_iteration': 400,
            'inner_iteration': 1,
            'sgd_iteration': 300,
            'prior': False,
            'prefix': result_folder,
            'display': True}

for m in method:
    for c in cost_type:
        hyperpara_dict = {'src_number': len(data_mc3['src_index']),
                          'tar_number': len(data_mc3['tar_index']),
                          'dimension': 20,
                          'loss_type': 'L2',
                          'cost_type': c,
                          'ot_method': m}

        gwd_model = GromovWassersteinLearning(hyperpara_dict)

        # initialize optimizer
        optimizer = optim.Adam(gwd_model.gwl_model.parameters(), lr=1e-3)
        scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=0.8)

        print("\nRunning Gromov-Wasserstein learning {}".format(data_name))

        # Gromov-Wasserstein learning
        time_start = time.time()
        gwd_model.train_without_prior(data_mc3, optimizer, opt_dict, scheduler=None)
        time_end = time.time()
        node_accuracy_GWEMBED[(n, i)].append(gwd_model.NC1)
        time_GWEMBED[(n, i)].append(time_end - time_start)
        print('Gromov-Wasserstein learning time cost: {:.4f}s'.format(time_end - time_start))

53
53
1113
1113

Running Gromov-Wasserstein learning syn_mc3_test_0
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/300: loss=129.617828.
inner 10/300: loss=127.218910.
inner 20/300: loss=123.890633.
inner 30/300: loss=119.626396.
inner 40/300: loss=114.539902.
inner 50/300: loss=108.822212.
inner 60/300: loss=102.694794.
inner 70/300: loss=96.421387.
inner 80/300: loss=90.321541.
inner 90/300: loss=84.718262.
inner 100/300: loss=79.840347.
inner 110/300: loss=75.783028.
inner 120/300: loss=72.541092.
inner 130/300: loss=70.051086.
inner 140/300: loss=68.213356.
inner 150/300: loss=66.905334.
inner 160/300: loss=65.997482.
inner 170/300: loss=65.372147.
inner 180/300: loss=64.937706.
inner 190/300: loss=64.630981.
inner 200/300: loss=64.410835.
inner 210/300: loss=64.250259.
inner 220/300: loss=64.131050.
inner 230/300: loss=64.040764.
inner 240/300: loss=63.970905.
inner 250/300: loss=63.915634.
inner 260/300: loss=63.870937.
inner 270/300

INFO:dev.util:Train Epoch: 0
INFO:dev.util:- node correctness: 92.4528%, 94.3396%
INFO:dev.util:- edge correctness: 96.8553%, 99.0566%
INFO:dev.util:- GW distance = 0.0005.


sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=43.410126.
inner 10/100: loss=43.181721.
inner 20/100: loss=43.005520.
inner 30/100: loss=42.960770.
inner 40/100: loss=42.949318.
inner 50/100: loss=42.938782.
inner 60/100: loss=42.928711.
inner 70/100: loss=42.920067.
inner 80/100: loss=42.912182.


INFO:dev.util:Train Epoch: 1
INFO:dev.util:- node correctness: 92.4528%, 98.1132%
INFO:dev.util:- edge correctness: 96.8553%, 99.3711%
INFO:dev.util:- GW distance = 0.0003.


inner 90/100: loss=42.904579.
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=35.513084.
inner 10/100: loss=35.329235.
inner 20/100: loss=35.196358.
inner 30/100: loss=35.166069.
inner 40/100: loss=35.155804.
inner 50/100: loss=35.144566.
inner 60/100: loss=35.134060.
inner 70/100: loss=35.124783.


...

inner 80/100: loss=35.116013.
inner 90/100: loss=35.107441.
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=34.944744.
inner 10/100: loss=34.823177.
inner 20/100: loss=34.737495.
inner 30/100: loss=34.714554.
inner 40/100: loss=34.702015.
inner 50/100: loss=34.689346.
inner 60/100: loss=34.678062.
inner 70/100: loss=34.667973.
inner 80/100: loss=34.658432.


INFO:dev.util:Train Epoch: 3
INFO:dev.util:- node correctness: 92.4528%, 96.2264%
INFO:dev.util:- edge correctness: 96.8553%, 100.0000%
INFO:dev.util:- GW distance = 0.0001.


inner 90/100: loss=34.649189.
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=36.780891.
inner 10/100: loss=36.725815.
inner 20/100: loss=36.688019.
inner 30/100: loss=36.674999.
inner 40/100: loss=36.664738.
inner 50/100: loss=36.654079.
inner 60/100: loss=36.643921.
inner 70/100: loss=36.634056.
inner 80/100: loss=36.624275.


INFO:dev.util:Train Epoch: 4
INFO:dev.util:- node correctness: 92.4528%, 98.1132%
INFO:dev.util:- edge correctness: 96.8553%, 99.3711%
INFO:dev.util:- GW distance = 0.0000.


inner 90/100: loss=36.614536.
Gromov-Wasserstein learning time cost: 5.4110s
